In [ ]:
# Hide non-critical library warnings so the experiment output stays readable.
# Remove these two lines when debugging, because warnings can contain useful information.
from warnings import simplefilter
simplefilter('ignore')

# NumPy is used to turn the model's raw scores (logits) into probabilities.
import numpy as np
# Llama is the llama.cpp Python interface used to run a GGUF model locally.
from llama_cpp import Llama

In [ ]:
# Download the model from Hugging Face on the first run and load it locally.
# Later runs normally reuse the model stored in the Hugging Face cache.
model = Llama.from_pretrained(
    # Hugging Face repository containing GGUF builds of Qwen3-0.6B.
    repo_id="Qwen/Qwen3-4B-GGUF",
    # Q8_0 is an approximately 8-bit quantized model: compact, but high quality.
    filename="Qwen3-4B-Q8_0.gguf",
    # The maximum prompt size is 512 tokens, which is enough for this example.
    n_ctx=512,
    # Retain raw vocabulary scores so we can inspect the final prompt position.
    logits_all=True,
    # Suppress llama.cpp diagnostic messages.
    verbose=False,
)

In [ ]:
# The model will choose a label token. These labels should each be one token.
labels = ["A", "B", 'C']
# Each choice corresponds by position to the label above: A, B, then C.
choices = ["Good", "Bad", "Sarcastically Bad"]
# This is the input that we want the local model to classify.
email = "Absolutely incredible. I’ve never paid this much to still be hungry."

# Pair the labels and choices to produce the multiple-choice text:
# A. Legitimate
# B. Spam
# C. Phishing
# strict=True catches a mistake if labels and choices have different lengths.
options = "\n".join(
    f"{label}. {choice}" for label, choice in zip(labels, choices, strict=True)
)

# Format the task using Qwen's special chat tokens. The system message gives
# the instruction, the user message supplies the email and choices, and the
# final assistant marker places the model exactly where its answer should begin.
# Supplying an empty <think> block asks Qwen3 to proceed directly to its answer.
prompt = f"""<|im_start|>system
Choose one option.<|im_end|>
<|im_start|>user
Food Review: {email}\n\n{options}<|im_end|>
<|im_start|>assistant
<think>\n\n</think>\n\n"""

print(prompt)

In [ ]:
# Convert the prompt from text to token IDs, then run one forward evaluation.
# add_bos=False avoids adding an extra beginning token because the chat prompt
# already has its own structure. special=True recognizes tokens such as
# <|im_start|> instead of treating them as ordinary text. No text is generated.
model.eval(model.tokenize(prompt.encode(), add_bos=False, special=True))

# Get the vocabulary-wide next-token scores after the last prompt token.
# model.n_tokens - 1 is the last position because Python indexes from zero.
logits = model.scores[model.n_tokens - 1]

# Translate A, B, and C into their vocabulary token IDs. [0] assumes each
# label is exactly one token, which is true for these labels with this model.
token_ids = [model.tokenize(label.encode(), add_bos=False)[0] for label in labels]

# Ignore the rest of the vocabulary and keep only the scores for A, B, and C.
choice_logits = np.asarray([logits[token_id] for token_id in token_ids])

# Apply a numerically stable softmax. logaddexp.reduce computes log(sum(exp(x))).
# The resulting values sum to 1 across these three choices. They are conditional
# choice probabilities, not necessarily calibrated real-world confidence scores.
probabilities = np.exp(choice_logits - np.logaddexp.reduce(choice_logits))

# Match each probability back to its human-readable choice and print it.
# :10 aligns the names; .1% formats 0.88 as 88.0%.
for choice, probability in zip(choices, probabilities, strict=True):
    print(f"{choice:10} {probability:.1%}")
